<link href="https://fonts.googleapis.com/css2?family=IBM+Plex+Sans&family=IBM+Plex+Serif&display=swap" rel="stylesheet">

<link href="https://fonts.googleapis.com/css2?family=Inter&family=Unica+One&display=swap" rel="stylesheet">
<span style="font-family: 'Unica One', sans-serif; font-size: 40px; color: #001219">
  <b>Neo4j + Python Case Study: The Habsburgs</b>
</span><br>
<span style="font-family: 'Inter', sans-serif; font-size: 16px; color: #005f73;">
  Prepared by Natalia Zelenko, powered by the beauty of graphs, 2025
</span>

<img src="pics/Habs_Victorian.png" alt="Victorian Habsburg Ad" width="350"/><br>
<span style="font-family: 'Inter', sans-serif; font-size: 10px;">
  Generated by Sora with a custom prompt.
</span>

In [1]:
from neo4j import GraphDatabase
import pandas as pd
from IPython.display import display, HTML

<span style="font-family: 'Inter', sans-serif; font-size: 18px; color: #ca6702;">
  Let's connect to Neo4j!
</span>

In [2]:
uri = "bolt://localhost:7687"
user = "neo4j"
password = "recursiveroyalty"

driver = GraphDatabase.driver(uri, auth=(user, password))

<span style="font-family: 'Inter', sans-serif; font-size: 18px; color: #ca6702;">
  Now let's run a query to test the connection.
</span>

In [3]:
with driver.session() as session:
    session.execute_read(
        lambda tx: (
            [print(record["message"]) for record in tx.run("RETURN 'Neo4j says hi!' AS message")]
        )
    )

Neo4j says hi!


<span style="font-family: 'Inter', sans-serif; font-size: 18px; color: #ca6702;">
  It worked! Now, what do we have here? Our little Habsburg cluster.
</span>

In [4]:
df_people = pd.read_csv("data/HABS_people.csv")
display(HTML(df_people.to_html()))

,person_id,name,born,died,gender,house
0,1,Charles II of Spain,1661,1700,M,Habsburg
1,2,Marie Louise d'Orléans,1662,1689,F,Orléans
2,3,Maria Anna of Neuburg,1667,1740,F,Wittelsbach
3,4,Philip IV of Spain,1605,1665,M,Habsburg
4,5,Mariana of Austria,1634,1696,F,Habsburg
5,6,Margaret Theresa of Spain,1651,1673,F,Habsburg
6,7,"Leopold I, Holy Roman Emperor",1640,1705,M,Habsburg
7,8,"Ferdinand III, Holy Roman Emperor",1608,1657,M,Habsburg
8,9,Maria Anna of Spain,1606,1646,F,Habsburg
9,10,Philip III of Spain,1578,1621,M,Habsburg


<span style="font-family: 'Inter', sans-serif; font-size: 18px; color: #ca6702;">
  Some of them are married.
</span>

In [5]:
df_marriage = pd.read_csv("data/HABS_marriage.csv")
display(HTML(df_marriage.to_html()))

,husband_id,wife_id,year
0,1,2,1679
1,1,3,1689
2,4,5,1649
3,7,6,1666
4,8,9,1631
5,10,11,1599
6,12,13,1570
7,14,15,1571
8,16,17,1600
9,18,19,1526


<span style="font-family: 'Inter', sans-serif; font-size: 18px; color: #ca6702;">
Some &mdash; related by blood.
</span>

In [6]:
df_blood = pd.read_csv("data/HABS_blood.csv")
display(HTML(df_blood.to_html()))

,person1_id,person2_id,person1_role,person2_role
0,4,5,uncle,niece
1,4,5,first cousin once removed,first cousin once removed
2,4,5,second cousin once removed,second cousin once removed
3,4,1,father,son
4,5,1,mother,son
5,4,1,great uncle,grand-nephew
6,5,1,cousin,cousin
7,4,6,father,daughter
8,5,6,mother,daughter
9,4,6,great uncle,grand-niece


<span style="font-family: 'Inter', sans-serif; font-size: 18px; color: #ca6702;">
  Wait. 25 people, 13 marriages and <i>HOW MANY</i> blood relationships?<br>
      What kind of a family <s>tree</s> spider web is this?
</span>

<img src="pics/Habs_50s.png" alt="50s Habsburg Ad" width="350"/><br>
<span style="font-family: 'Inter', sans-serif; font-size: 10px;">
  Generated by Sora with a custom prompt.
</span>

<span style="font-family: 'Inter', sans-serif; font-size: 18px; color: #ca6702;">
  This function will help us create a Neo4j node from one row of our df_people DataFrame.
</span>

In [7]:
def create_person_node(tx, person_id, name, born, died, gender, house):
    query = """
    MERGE (p:Person {person_id: $person_id})
    ON CREATE SET
        p.name = $name,
        p.born = $born,
        p.died = $died,
        p.gender = $gender,
        p.house = $house
        RETURN CASE WHEN p.name IS NOT NULL THEN 1 ELSE 0 END AS processed_flag
    """
    result = tx.run(query, person_id=person_id, name=name, born=born, died=died, gender=gender, house=house)
    record = result.single()
    return record["processed_flag"] if record else 0

<span style="font-family: 'Inter', sans-serif; font-size: 18px; color: #ca6702;">
  Now let's use it.
</span>

In [8]:
total_processed = 0
with driver.session() as session:
    for _, row in df_people.iterrows():
        processed = session.execute_write(
            create_person_node,
            int(row['person_id']),
            row['name'],
            int(row['born']) if not pd.isna(row['born']) else None,
            int(row['died']) if not pd.isna(row['died']) else None,
            row['gender'],
            row['house']
        )
        total_processed += processed

print(f"Total Person nodes processed: {total_processed}")

Total Person nodes processed: 25


<span style="font-family: 'Inter', sans-serif; font-size: 18px; color: #ca6702;">
  Here they are, lonely unconnected nodes. Note that the orange ones are all Habsburgs.
</span>
<img src="screenshots/scr_001.png" alt="25 lonely persons nodes" width="600"/><br>
<span style="font-family: 'Inter', sans-serif; font-size: 8px;">
  screenshot from Neo4j Bloom
</span>

<span style="font-family: 'Inter', sans-serif; font-size: 18px; color: #ca6702;">
  And here's everyone's favourite Habsburg.
</span>
<img src="screenshots/scr_002.png" alt="25 lonely persons nodes" width="500"/><br>
<span style="font-family: 'Inter', sans-serif; font-size: 8px;">
  screenshot from Neo4j Bloom
</span>

<span style="font-family: 'Inter', sans-serif; font-size: 18px; color: #ca6702;">
  <br>Now let's define a similar function to create the bonds of matrimony, and then use it.
</span>

In [9]:
def create_marriage_relationship(tx, husband_id, wife_id, year):
    query = """
    MATCH (h:Person {person_id: $husband_id})
    MATCH (w:Person {person_id: $wife_id})
    MERGE (h)-[r:MARRIED_TO]->(w)
    ON CREATE SET r.year = $year
    RETURN CASE WHEN r.year IS NOT NULL THEN 1 ELSE 0 END AS processed_flag
    """
    result = tx.run(query, husband_id=husband_id, wife_id=wife_id, year=year)
    record = result.single()
    return record["processed_flag"] if record else 0

In [10]:
total_processed = 0
with driver.session() as session:
    for _, row in df_marriage.iterrows():
        processed = session.execute_write(
            create_marriage_relationship,
            int(row['husband_id']),
            int(row['wife_id']),
            int(row['year'])
        )
        total_processed += processed

print(f"Total MARRIED_TO relationships processed: {total_processed}")

Total MARRIED_TO relationships processed: 13


<span style="font-family: 'Inter', sans-serif; font-size: 18px; color: #ca6702;">
  Now, every one's coupled up nicely.
</span>
<img src="screenshots/scr_003.png" alt="not-lonely-anymore persons nodes" width="600"/><br>
<span style="font-family: 'Inter', sans-serif; font-size: 8px;">
  screenshot from Neo4j Bloom
</span>

<img src="screenshots/scr_004.png" alt="a marriage info" width="500"/><br>
<span style="font-family: 'Inter', sans-serif; font-size: 8px;">
  screenshot from Neo4j Bloom
</span>

<span style="font-family: 'Inter', sans-serif; font-size: 18px; color: #ca6702;">
  <br>Ready or not, here they come! The blood relationships.
</span>

In [11]:
def create_blood_relationship(tx, person1_id, person2_id, person1_role, person2_role):
    rel_type = person1_role.strip().upper().replace(" ", "_").replace("-", "_")

    query = f"""
    MATCH (p1:Person {{person_id: $person1_id}})
    MATCH (p2:Person {{person_id: $person2_id}})
    MERGE (p1)-[r:{rel_type}]->(p2)
    ON CREATE SET r.to_role = $person2_role
    RETURN CASE WHEN r.to_role IS NOT NULL THEN 1 ELSE 0 END AS processed_flag
    """
    result = tx.run(query, person1_id=person1_id, person2_id=person2_id, person2_role=person2_role)
    record = result.single()
    return record["processed_flag"] if record else 0

In [12]:
total_processed = 0
with driver.session() as session:
    for _, row in df_blood.iterrows():
        processed = session.execute_write(
            create_blood_relationship,
            int(row['person1_id']),
            int(row['person2_id']),
            row['person1_role'],
            row['person2_role']
        )
        total_processed += processed

print(f"Total blood relationships processed: {total_processed}")

Total blood relationships processed: 62


<span style="font-family: 'Inter', sans-serif; font-size: 18px; color: #ca6702;">
  Wasn't that supposed to be a tree? Nevermind.
</span>
<img src="screenshots/scr_005.png" alt="the tree"/><br><span style="font-family: 'Inter', sans-serif; font-size: 8px;">
  screenshot from Neo4j Bloom
</span><br>

<br><img src="pics/Habs_80s.png" alt="80s Habsburg Ad" width="350"/><br>
<span style="font-family: 'Inter', sans-serif; font-size: 10px;">
  Generated by Sora with a custom prompt.
</span>

<span style="font-family: 'Inter', sans-serif; font-size: 18px; color: #ca6702;">
  <br>Now let's make things a little simpler.<br>First, let's define some functions: one to delete relationships, and one to rename them.
</span>

In [13]:
def rename_relationship_type(tx, old_type, new_type, new_to_role_value):
    query = f"""
    MATCH (a)-[r:{old_type}]->(b)
    CALL (a, b, r) {{
      WITH a, b, r
      MERGE (a)-[new_r:{new_type}]->(b)
      SET new_r += properties(r)
      SET new_r.to_role = $new_to_role
      DELETE r
    }}
    RETURN COUNT(*) AS updated
    """
    result = tx.run(query, new_to_role=new_to_role_value)
    summary = result.single()
    print(f"Renamed {old_type} → {new_type}, updated: {summary['updated']} relationship(s).")

In [14]:
def delete_relationship_type(tx, rel_type):
    query = f"""
    MATCH ()-[r:{rel_type}]->()
    DELETE r
    RETURN COUNT(r) AS deleted
    """
    result = tx.run(query)
    summary = result.single()
    print(f"Deleted {summary['deleted']} relationship(s) of type {rel_type}.")

<span style="font-family: 'Inter', sans-serif; font-size: 18px; color: #ca6702;">
  <br>Now let's rename "cousins" to "first cousins" and delete "second cousins once removed".
</span>

In [15]:
with driver.session() as session:
    session.execute_write(
        rename_relationship_type,
        "COUSIN",
        "FIRST_COUSIN", 
        "first cousin"  
    )
    session.execute_write(delete_relationship_type,"SECOND_COUSIN_ONCE_REMOVED")

Renamed COUSIN → FIRST_COUSIN, updated: 7 relationship(s).
Deleted 2 relationship(s) of type SECOND_COUSIN_ONCE_REMOVED.


<span style="font-family: 'Inter', sans-serif; font-size: 18px; color: #ca6702;">
  <br>Let's just count both "first cousins once removed" and "second cousins" as "distant cousins".
</span>

In [16]:
with driver.session() as session:
    session.execute_write(rename_relationship_type,
                          "FIRST_COUSIN_ONCE_REMOVED", "DISTANT_COUSIN", "distant cousin")
    session.execute_write(rename_relationship_type,
                          "SECOND_COUSIN", "DISTANT_COUSIN", "distant cousin")

Renamed FIRST_COUSIN_ONCE_REMOVED → DISTANT_COUSIN, updated: 9 relationship(s).
Renamed SECOND_COUSIN → DISTANT_COUSIN, updated: 3 relationship(s).


In [17]:
with driver.session() as session:
    result = session.run("MATCH ()-[r:DISTANT_COUSIN]->() RETURN count(r) AS count")
    record = result.single()
    count = record["count"] if record else 0
    print(f"Number of DISTANT_COUSIN relationships: {count}")

Number of DISTANT_COUSIN relationships: 10


<span style="font-family: 'Inter', sans-serif; font-size: 18px; color: #ca6702;">
  <br>Now let’s say we’ve realized how dumb we were not to tag blood vs. marriage relationships back when we were parsing our data.<br>
  Well, better late than never.
</span>

In [18]:
def tag_marriage_relationships(tx):
    query = """
    MATCH ()-[r:MARRIED_TO]->()
    SET r.type = "by marriage"
    RETURN COUNT(r) AS updated
    """
    result = tx.run(query)
    print(f"Tagged {result.single()['updated']} relationships as 'by marriage'.")

In [19]:
def tag_blood_relationships(tx):
    query = """
    MATCH ()-[r]->()
    WHERE type(r) <> "MARRIED_TO"
    SET r.type = "by blood"
    RETURN COUNT(r) AS updated
    """
    result = tx.run(query)
    print(f"Tagged {result.single()['updated']} relationships as 'by blood'.")

In [20]:
with driver.session() as session:
    session.execute_write(tag_marriage_relationships)
    session.execute_write(tag_blood_relationships)

Tagged 13 relationships as 'by marriage'.
Tagged 58 relationships as 'by blood'.


<span style="font-family: 'Inter', sans-serif; font-size: 18px; color: #ca6702;">
  <br>Time to <s>party</s> query, y'all!
</span>

<span style="font-family: 'Inter', sans-serif; font-size: 18px; color: #ca6702;">
  Who's that guy again, the son of Philip IV?
</span>

In [21]:
with driver.session() as session:
    result = session.run("""
        MATCH (father:Person)
        WHERE father.name CONTAINS "Philip IV"
        MATCH (father)-[:FATHER]->(child:Person)
        WHERE child.gender = "M"
        RETURN child
        LIMIT 1
    """)
    record = result.single()
    if record:
        child_node = record["child"]
        print("Raw returned node object:", child_node)
        print("Node properties:", dict(child_node))
    else:
        print("No matching child found.")

Raw returned node object: <Node element_id='4:73115f1c-0399-48a2-8a03-8b2e4c96d2a1:23' labels=frozenset({'Person'}) properties={'gender': 'M', 'born': 1661, 'name': 'Charles II of Spain', 'died': 1700, 'house': 'Habsburg', 'person_id': 1}>
Node properties: {'gender': 'M', 'born': 1661, 'name': 'Charles II of Spain', 'died': 1700, 'house': 'Habsburg', 'person_id': 1}


<span style="font-family: 'Inter', sans-serif; font-size: 18px; color: #ca6702;">
  <br>Who's one of the most "connected"? *wink-wink*
</span>

In [22]:
with driver.session() as session:
    query = """
    MATCH (p:Person)
    MATCH (p)-[r]-()
    WITH p, count(r) AS relCount
    ORDER BY relCount DESC
    LIMIT 1
    MATCH (p)-[r2]-(connected)
    RETURN p, collect({
      relationship: type(r2),
      node: connected,
      direction: CASE WHEN startNode(r2) = p THEN "outgoing" ELSE "incoming" END
    }) AS connections
    """
    result = session.run(query)
    record = result.single()
    if record:
        person = record["p"]
        connections = record["connections"]
        print(f"Person: {person['name']} (person_id: {person['person_id']})")
        print("Connections:")
        for c in connections:
            node = c['node']
            rel_type = c['relationship']
            direction = c['direction']
            arrow = "→" if direction == "outgoing" else "←"
            print(f" - {rel_type} {arrow} {node.get('name', 'Unknown')}")
    else:
        print("No person found.")


Person: Philip IV of Spain (person_id: 4)
Connections:
 - MARRIED_TO → Mariana of Austria
 - UNCLE → Mariana of Austria
 - FATHER → Margaret Theresa of Spain
 - FATHER → Charles II of Spain
 - FATHER ← Philip III of Spain
 - MOTHER ← Margaret of Austria
 - GREAT_UNCLE → Margaret Theresa of Spain
 - GREAT_UNCLE → Charles II of Spain
 - DISTANT_COUSIN → Mariana of Austria
 - DISTANT_COUSIN → Margaret of Austria


<span style="font-family: 'Inter', sans-serif; font-size: 18px; color: #ca6702;">
  <br>How is that "handsome" guy connected to our Charlie?
</span>

In [23]:
with driver.session() as session:
    result = session.run(
        """
        MATCH (a:Person), (b:Person)
        WHERE a.name CONTAINS $nameA AND b.name CONTAINS $nameB
        MATCH path = shortestPath((a)-[*]-(b))
        RETURN path
        LIMIT 1
        """,
        nameA="Charles II of Spain",
        nameB="Handsome"
    )
    record = result.single()
    if not record:
        print(f"No path found between these persons.")
    else:
        path = record["path"]
        nodes = path.nodes
        relationships = path.relationships

        print(f"Shortest path between the two:")
        for i in range(len(nodes)):
            node = nodes[i]
            print(f"Node {i + 1}: {node['name']} (ID: {node['person_id']})")
            if i < len(relationships):
                rel = relationships[i]
                print(f"  --[{rel.type}]--")


Shortest path between the two:
Node 1: Charles II of Spain (ID: 1)
  --[FATHER]--
Node 2: Philip IV of Spain (ID: 4)
  --[FATHER]--
Node 3: Philip III of Spain (ID: 10)
  --[FATHER]--
Node 4: Philip II of Spain (ID: 12)
  --[FATHER]--
Node 5: Charles V, Holy Roman Emperor (ID: 18)
  --[FATHER]--
Node 6: Philip the Handsome (ID: 24)


<span style="font-family: 'Inter', sans-serif; font-size: 18px; color: #ca6702;">
  <br>And finally...
</span>

In [24]:
with driver.session() as session:
    query = """
    MATCH (a)-[:MARRIED_TO]->(b)
    WITH DISTINCT a, b
    WITH collect({a: a, b: b}) AS marriedPairs, size(collect(DISTINCT [a,b])) AS total_married
    UNWIND marriedPairs AS pair
    WITH pair.a AS a, pair.b AS b, total_married
    WHERE EXISTS {
      MATCH (a)-[r]->(b)
      WHERE r.type = "by blood"
    } OR EXISTS {
      MATCH (b)-[r]->(a)
      WHERE r.type = "by blood"
    }
    RETURN
      total_married,
      count(*) AS also_blood_related,
      round(100.0 * count(*) / total_married, 2) AS percent_close_kin
    """
    result = session.run(query)
    stats = result.single()
    print(f"Total marriages: {stats['total_married']}")
    print(f"Intra-family marriages: {stats['also_blood_related']}")
    print(f"Genetic intimacy rate: {stats['percent_close_kin']}%")
    

Total marriages: 13
Intra-family marriages: 7
Genetic intimacy rate: 53.85%


<span style="font-family: 'Inter', sans-serif; font-size: 18px; color: #ca6702;">
  <br>P.S. Let's highlight <i>something</i> by creating new types of relationships based on the existing ones.
</span>

In [25]:
def create_combined_relationship(tx, rel1, rel2, role_to_value):
    new_rel_type = f"{rel1}_AND_{rel2}"

    query = f"""
    MATCH (a)-[r1:{rel1}]->(b)
    MATCH (a)-[r2:{rel2}]->(b)
    WHERE NOT EXISTS {{
      MATCH (a)-[:{new_rel_type}]->(b)
    }}
    CREATE (a)-[new_r:{new_rel_type} {{role_to: $role_to}}]->(b)
    RETURN COUNT(*) AS created_count
    """

    result = tx.run(query, role_to=role_to_value)
    count = result.single()["created_count"]
    print(f"Created {count} '{new_rel_type}' relationships with role_to='{role_to_value}'.")

In [26]:
with driver.session() as session:
    session.execute_write(create_combined_relationship, "UNCLE", "MARRIED_TO", "niece and married to")
    session.execute_write(create_combined_relationship, "MOTHER", "FIRST_COUSIN", "child and cousin")

Created 3 'UNCLE_AND_MARRIED_TO' relationships with role_to='niece and married to'.
Created 3 'MOTHER_AND_FIRST_COUSIN' relationships with role_to='child and cousin'.


<span style="font-family: 'Inter', sans-serif; font-size: 18px; color: #ca6702;">
  Oh, isn't this the sweetest little group!
</span>
<img src="screenshots/scr_006.png" alt="the closest"/><br><span style="font-family: 'Inter', sans-serif; font-size: 8px;">
  screenshot from Neo4j Bloom
</span>

<span style="font-family: 'Inter', sans-serif; font-size: 18px; color: #ca6702;">
  <br>Never forget to close the driver.
</span>

In [27]:
driver.close()

<img src="pics/title.png" alt="title" width=800/><br>
<span style="font-family: 'Inter', sans-serif; font-size: 16px;">
  <i>The Graph Witch Luring the Developer Away from Tables</i>
</span><br>
<span style="font-family: 'Inter', sans-serif; font-size: 10px;">
  Generated by Sora with a custom prompt.
</span>

<span style="font-family: 'Unica One', sans-serif; font-size: 40px; color: #005f73;">
  <b>the end</b>
</span>